# ADALINE y Cancelación de Eco con Filtro FIR Adaptativo (LMS)
### Notebook didáctico para clase

**Contenido del notebook:**
- Generación de señal con ruido y eco
- Implementación de ADALINE como filtro adaptativo
- Regla de aprendizaje LMS
- Cancelación de ruido
- Cancelación de eco real con filtro FIR adaptativo
- Animación del aprendizaje
- Sliders interactivos para el learning rate

*Generado automáticamente: 2026-03-04*

## 1. Concepto

ADALINE puede utilizarse como **filtro adaptativo** para cancelar ruido o eco.

Modelo de señal observada:

\[ d(n) = s(n) + v(n) \]

donde:
- `s(n)` = señal deseada
- `v(n)` = ruido o eco
- `d(n)` = señal observada

La red ADALINE aprende a estimar el ruido.

Regla LMS:

\[ w(n+1) = w(n) + \eta e(n)x(n) \]


In [ ]:
# Librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from ipywidgets import interact, FloatSlider

## 2. Generación de señal con ruido

In [ ]:
np.random.seed(0)

N = 400
t = np.arange(N)

# señal limpia
signal = np.sin(0.05*np.pi*t)

# ruido
noise = 0.5*np.random.randn(N)

# señal observada
d = signal + noise

plt.figure()
plt.plot(signal,label='Señal original')
plt.plot(d,label='Señal con ruido')
plt.legend()
plt.grid()
plt.title('Señal con ruido')


## 3. ADALINE como filtro adaptativo

In [ ]:
def adaline_filter(x, d, lr=0.01):

    w = 0
    y = np.zeros(len(x))
    e = np.zeros(len(x))

    for n in range(len(x)):

        y[n] = w * x[n]
        e[n] = d[n] - y[n]

        w = w + lr * e[n] * x[n]

    return y,e


## 4. Cancelación de ruido

In [ ]:
y,e = adaline_filter(noise,d,0.01)

plt.figure()
plt.plot(signal,label='Señal real')
plt.plot(e,label='Salida filtrada')
plt.legend()
plt.grid()
plt.title('Cancelación de ruido con ADALINE')


## 5. Evolución del error LMS

In [ ]:
plt.figure()
plt.plot(e**2)
plt.grid()
plt.title('Error cuadrático durante el aprendizaje')
plt.xlabel('Iteración')


## 6. Slider interactivo para learning rate

In [ ]:
def simulate(lr):

    y,e = adaline_filter(noise,d,lr)

    plt.figure()
    plt.plot(signal,label='Señal real')
    plt.plot(e,label='Señal filtrada')
    plt.legend()
    plt.grid()
    plt.title(f'Learning Rate = {lr}')

interact(simulate,
         lr=FloatSlider(min=0.001,max=0.1,step=0.005,value=0.01))


## 7. Cancelación de eco real con Filtro FIR Adaptativo

Modelo de eco:

\[ y(n) = \sum w_k x(n-k) \]


In [ ]:
def adaptive_fir(x, d, M=5, lr=0.01):

    w = np.zeros(M)
    y = np.zeros(len(x))
    e = np.zeros(len(x))

    for n in range(M,len(x)):

        x_vec = x[n-M:n]

        y[n] = np.dot(w,x_vec)

        e[n] = d[n] - y[n]

        w = w + lr * e[n] * x_vec

    return y,e


## 8. Simulación de eco

In [ ]:
echo = np.roll(signal,20)*0.7

mic = signal + echo

plt.figure()
plt.plot(mic,label='Micrófono con eco')
plt.plot(signal,label='Señal original')
plt.legend()
plt.grid()


## 9. Cancelación de eco con LMS

In [ ]:
y,e = adaptive_fir(signal,mic,M=10,lr=0.01)

plt.figure()
plt.plot(signal,label='Señal original')
plt.plot(e,label='Señal sin eco')
plt.legend()
plt.grid()
plt.title('Cancelación de eco con filtro adaptativo LMS')


## 10. Animación del aprendizaje LMS

In [ ]:
M=8
lr=0.01

w=np.zeros(M)
y=np.zeros(N)
e=np.zeros(N)

fig,ax=plt.subplots()

line,=ax.plot([],[])

ax.set_xlim(0,N)
ax.set_ylim(-2,2)
ax.grid()

def update(n):

    global w

    if n>M:

        x_vec=signal[n-M:n]

        y[n]=np.dot(w,x_vec)

        e[n]=mic[n]-y[n]

        w=w+lr*e[n]*x_vec

    line.set_data(range(n),e[:n])

    return line,

ani=FuncAnimation(fig,update,frames=N,interval=30)

plt.show()
